## 3DoF Entry VTOL w/o Aoa SCP

Imports

In [1]:
# Basic imports
import importlib
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Trajopt imports --> pip install -e ~/ACL/trajopt
import trajopt; importlib.reload(trajopt)
import trajopt.core.modules.method.scp as scp
import trajopt.core.problem as prob
import trajopt.utils.config_loader as cfg
import trajopt.utils.tools as tools
import trajopt.analysis.default_analysis as default_analysis
import trajopt.analysis.statistics as stats
import copy

setup problem and run SCP

In [ ]:
from typing import Any

def add_monte_carlo_dispersions(mission_dict, realization):
    for mc_var, mc_disp in realization.items():
        mission_dict[mc_var] = mission_dict[mc_var] + mc_disp

gen_mc_variations    = 1
save_mc_variations   = 0
save_scenario_data   = 0

example_name = "vtol1_entry_3dof"
mc_name = "mc1"
scenario_data_name = example_name

nominal_config  = cfg.load_configs(example_name)

mv_variations = cfg.load_mv_variations(example_name)

if gen_mc_variations:
    mc_variations = cfg.gen_mc_variations(example_name)

    if save_mc_variations:
        np.save(f"data/mc_variations/{mc_name}", mc_variations)
else:
    mc_variations = np.load(f"data/mc_variations/{mc_name}.npy", allow_pickle=True).item()

variations = {
    "method": mv_variations,
    "mission": mc_variations
}

scenario_data = {}

# loop through method variations
for name, method_variation in variations["method"].items():
    
    # initialize method sub-dictionary for scenario_data dict
    scenario_data[name] = {"method_params": {},
                                  'mc_data': [None] * (variations["mission"]["num_variations"] + 1),
                                  }

    cached_subprob = None
    
    # loop through monte-carlo mission parameter realizations (number of runs)
    for run_idx, realization in enumerate(variations["mission"]["realizations"]):
        
        # take in nominal configs
        run_config = copy.deepcopy(nominal_config)

        # set method variations
        run_config["method"] = tools.deep_update(run_config["method"], method_variation)

        # set monte carlo mission variations
        add_monte_carlo_dispersions(run_config["mission"], realization)

        # create problem instance
        problem = prob.Problem(run_config, cached_subprob)
        
        # run SCP
        problem = scp.run_scp(problem)

        # perform default analysis on this mc run and store related params
        scenario_data[name]["mc_data"][run_idx] = default_analysis.perform_default_analysis(problem)

        # # store total time for scp (used to calculate time to converge)
        # scenario_data[name]['mc_data'][run_idx] = problem.solution['t_all']
        
        # cache subproblem graph to speed up solves
        cached_subprob = None # problem.method.subprob

if save_scenario_data:
    np.save(f"data/scenario_data/{scenario_data_name}_{mc_name}", scenario_data)

scales: 
d: 6378137.0000, t: 806.3293, m: 104305.0000, v: 7910.0900, a: 9.8100, f: 1023232.0500, ang: 57.2958, angv: 0.0711
Initial guess time: 0.3149438329273835 seconds
--------------------------------------------------------------------------------------------------------------------------------------------------------
                                              ..:: vtol: PTR with Virtual Buffer ::..
--------------------------------------------------------------------------------------------------------------------------------------------------------
  Iteration |  Propagation |   Solve   |    Parse   |  log(dz)  |      log(VB)    |   log(VB)   |  log(VB)    | Solve status |  Time of    |   Cost    
            |   time [ms]  | time [ms] |  time [ms] |           |  (path + NFZ)   |  (terminal) |  (dynamics) |              |  Flight [s] |           
-------------------------------------------------------------------------------------------------------------------------------------

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/cvxpy/reductions/solvers/solving_chain.py:250: UserWarning: Your problem has too many parameters for efficient DPP compilation. We suggest setting 'ignore_dpp = True'.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/cvxpy/reductions/solvers/solving_chain_utils.py:41: UserWarning: The problem has an expression with dimension greater than 2. Defaulting to the SCIPY backend for canonicalization.
  warnings.warn(UserWarning(


     01     |    02378.3   |   0008.8  |   10441.2   |   -2.2    |      -12.0      |    -01.2    |     -12.0   |    optimal    |   1789.17   |  559.6
     02     |    00012.3   |   0010.0  |   0001.1   |   -2.8    |      -12.0      |    -01.5    |     -12.0   |    optimal    |   1629.34   |  950.0
     03     |    00008.7   |   0010.0  |   0001.0   |   -2.9    |      -12.0      |    -02.6    |     -12.0   |    optimal    |   1556.66   |  1112.6
     04     |    00008.3   |   0009.8  |   0001.0   |   -3.1    |      -12.0      |    -02.7    |     -12.0   |    optimal    |   1582.54   |  972.2
     05     |    00008.6   |   0008.3  |   0000.9   |   -3.2    |      -12.0      |    -02.7    |     -12.0   |    optimal    |   1600.88   |  905.9
     06     |    00007.5   |   0008.0  |   0000.9   |   -3.7    |      -12.0      |    -02.8    |     -12.0   |    optimal    |   1620.23   |  837.8
     07     |    00008.6   |   0010.3  |   0001.0   |   -3.6    |      -12.0      |    -02.8    |     -1

make plots

In [ ]:
print(scenario_data.keys())